# Tech Challenge Fase 3
## Notebook 00 - Setup do Ambiente AWS S3 + Databricks + Unity Catalog

### Objetivo

Este notebook prepara a base técnica da **Fase 3 - Predição e Inteligência Analítica para Alfabetização no Brasil**.

Ele cria uma área independente para Ciência de Dados e Machine Learning e conecta essa nova fase, em modo de leitura, aos dados analíticos produzidos na Fase 2.

### Separação arquitetural

| Papel | Caminho AWS S3 | Permissão esperada |
|---|---|---|
| Fonte oficial | `s3://s3tc2/projetos/fiap/tech_challenge_fase2/gold/` | Somente leitura |
| Projeto Fase 3 | `s3://s3tc2/projetos/fiap/tech_challenge_fase3/` | Leitura e escrita |

> **Regra de segurança:** nenhuma célula deste notebook grava, move, renomeia ou exclui arquivos no diretório Gold da Fase 2.

### Fluxo

```text
Gold Fase 2 (imutável)
        ↓
Validação e inventário
        ↓
data/input_gold (referência lógica)
        ↓
EDA e preparação
        ↓
Treino / validação / teste
        ↓
Modelos, métricas, relatórios e imagens
```


## 1. Validação do External Volume

O bucket `s3://s3tc2/` já está governado pelo Unity Catalog por meio do volume `workspace.default.vol_trio_drive`.

Reutilizamos o mesmo volume e criamos apenas um novo prefixo lógico. Isso evita conflitos de `LOCATION_OVERLAP` e mantém as fases isoladas.


In [0]:
%sql
SHOW VOLUMES IN workspace.default;


## 2. Conferência da localização física do volume

O resultado deve indicar uma localização associada a `s3://s3tc2/`.


In [0]:
%sql
DESCRIBE VOLUME workspace.default.vol_trio_drive;


## 3. Configuração central de caminhos

Usaremos caminhos `/Volumes/...` nos notebooks, pois eles preservam a governança do Unity Catalog. Os endereços `s3://...` permanecem documentados para rastreabilidade arquitetural.

As constantes da Fase 2 recebem o prefixo `SOURCE_`; as da Fase 3 recebem `PROJECT_` ou descrevem sua finalidade. Essa convenção reduz o risco de uma gravação acidental na fonte.


In [0]:
%python
from datetime import datetime, timezone
import json
from pyspark.sql.types import StructType, StructField, StringType, LongType

PROJECT_NAME = "fiap_alfabetizacao_ml"
PROJECT_VERSION = "tech_challenge_fase3"
ENVIRONMENT = "dev"

VOLUME_CATALOG = "workspace"
VOLUME_SCHEMA = "default"
VOLUME_NAME = "vol_trio_drive"
EXTERNAL_VOLUME = f"{VOLUME_CATALOG}.{VOLUME_SCHEMA}.{VOLUME_NAME}"
VOLUME_ROOT = f"/Volumes/{VOLUME_CATALOG}/{VOLUME_SCHEMA}/{VOLUME_NAME}"

# Fonte oficial e imutável: Gold construída na Fase 2.
SOURCE_FASE2_PREFIX = "projetos/fiap/tech_challenge_fase2/gold"
SOURCE_GOLD_PATH = f"{VOLUME_ROOT}/{SOURCE_FASE2_PREFIX}"
SOURCE_GOLD_S3 = "s3://s3tc2/projetos/fiap/tech_challenge_fase2/gold/"

# Área exclusiva de escrita da Fase 3.
PROJECT_PREFIX = "projetos/fiap/tech_challenge_fase3"
BASE_PATH = f"{VOLUME_ROOT}/{PROJECT_PREFIX}"
PROJECT_S3 = "s3://s3tc2/projetos/fiap/tech_challenge_fase3/"

DATA_PATH = f"{BASE_PATH}/data"
NOTEBOOKS_PATH = f"{BASE_PATH}/notebooks"
SRC_PATH = f"{BASE_PATH}/src"
REPORTS_PATH = f"{BASE_PATH}/reports"
IMAGES_PATH = f"{BASE_PATH}/images"
MODELS_PATH = f"{BASE_PATH}/models"
ARTIFACTS_PATH = f"{BASE_PATH}/artifacts"
LOGS_PATH = f"{BASE_PATH}/logs"
CONFIG_PATH = f"{BASE_PATH}/config"

INPUT_GOLD_PATH = f"{DATA_PATH}/input_gold"
EXTERNAL_DATA_PATH = f"{DATA_PATH}/external"
INTERIM_DATA_PATH = f"{DATA_PATH}/interim"
PROCESSED_DATA_PATH = f"{DATA_PATH}/processed"
TRAIN_DATA_PATH = f"{PROCESSED_DATA_PATH}/train"
VALIDATION_DATA_PATH = f"{PROCESSED_DATA_PATH}/validation"
TEST_DATA_PATH = f"{PROCESSED_DATA_PATH}/test"

EXECUTION_TS_UTC = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
EXECUTION_DATE = datetime.now(timezone.utc).strftime("%Y-%m-%d")

print(f"Projeto: {PROJECT_NAME}")
print(f"Versão: {PROJECT_VERSION}")
print(f"Fonte Gold Fase 2 (leitura): {SOURCE_GOLD_PATH}")
print(f"Destino Fase 3 (escrita): {BASE_PATH}")
print(f"Execução UTC: {EXECUTION_TS_UTC}")


## 4. Verificação de segurança dos caminhos

Antes de qualquer criação ou gravação, validamos que:

- a fonte termina em `tech_challenge_fase2/gold`;
- o destino pertence à `tech_challenge_fase3`;
- fonte e destino são distintos;
- nenhum caminho gravável está dentro da Gold da Fase 2.

Se alguma regra falhar, o notebook é interrompido imediatamente.


In [0]:
%python
def normalizar_path(path: str) -> str:
    return path.rstrip("/")

source_normalized = normalizar_path(SOURCE_GOLD_PATH)
destination_normalized = normalizar_path(BASE_PATH)

assert source_normalized.endswith("/tech_challenge_fase2/gold"),     "SOURCE_GOLD_PATH não aponta para a Gold oficial da Fase 2."
assert "/tech_challenge_fase3" in destination_normalized,     "BASE_PATH deve pertencer exclusivamente à Fase 3."
assert source_normalized != destination_normalized,     "Fonte e destino não podem ser o mesmo caminho."
assert not destination_normalized.startswith(source_normalized + "/"),     "O destino da Fase 3 não pode estar dentro da Gold da Fase 2."

print("Verificação de caminhos validado com sucesso.")


## 5. Estrutura de diretórios da Fase 3

A estrutura atende ao mínimo solicitado no desafio e separa dados, código, modelos e evidências.

```text
tech_challenge_fase3/
├── data/
│   ├── input_gold/          # referência lógica; não duplica a Gold automaticamente
│   ├── external/            # enriquecimentos autorizados
│   ├── interim/             # resultados temporários reproduzíveis
│   └── processed/
│       ├── train/
│       ├── validation/
│       └── test/
├── notebooks/
├── src/
│   ├── preprocessing/
│   ├── modeling/
│   ├── evaluation/
│   └── visualization/
├── models/
├── artifacts/
│   ├── metrics/
│   ├── feature_importance/
│   ├── shap/
│   └── predictions/
├── reports/
├── images/
├── logs/
└── config/
```

Os arquivos `README.md`, `requirements.txt` e `.gitignore` pertencem ao repositório Git. Seus nomes serão registrados na configuração, mas o setup não substitui o versionamento do código por armazenamento no S3.


In [0]:
%python
directories = [
    DATA_PATH,
    INPUT_GOLD_PATH,
    EXTERNAL_DATA_PATH,
    INTERIM_DATA_PATH,
    PROCESSED_DATA_PATH,
    TRAIN_DATA_PATH,
    VALIDATION_DATA_PATH,
    TEST_DATA_PATH,
    NOTEBOOKS_PATH,
    f"{SRC_PATH}/preprocessing",
    f"{SRC_PATH}/modeling",
    f"{SRC_PATH}/evaluation",
    f"{SRC_PATH}/visualization",
    MODELS_PATH,
    f"{ARTIFACTS_PATH}/metrics",
    f"{ARTIFACTS_PATH}/feature_importance",
    f"{ARTIFACTS_PATH}/shap",
    f"{ARTIFACTS_PATH}/predictions",
    REPORTS_PATH,
    IMAGES_PATH,
    f"{LOGS_PATH}/setup",
    f"{LOGS_PATH}/data_quality",
    f"{LOGS_PATH}/training",
    f"{LOGS_PATH}/evaluation",
    CONFIG_PATH,
]

for directory in directories:
    if normalizar_path(directory).startswith(source_normalized):
        raise ValueError(f"Gravação bloqueada na fonte Gold: {directory}")
    dbutils.fs.mkdirs(directory)

print(f"{len(directories)} diretórios criados ou validados na área da Fase 3.")


## 6. Validação de acesso à Gold da Fase 2

Esta etapa apenas lista os objetos existentes. Ela não copia e não altera a fonte.

> No S3, diretórios são prefixos. Por isso, a existência da raiz Gold é comprovada pela listagem de seus objetos ou subdiretórios.


In [0]:
%python
try:
    source_items = dbutils.fs.ls(SOURCE_GOLD_PATH)
except Exception as exc:
    raise RuntimeError(
        "Não foi possível ler a Gold da Fase 2. Verifique o External Volume, "
        "as permissões READ VOLUME e o caminho configurado."
    ) from exc

if not source_items:
    raise ValueError(f"A Gold da Fase 2 está acessível, porém vazia: {SOURCE_GOLD_PATH}")

print(f"Gold acessível. Itens encontrados na raiz: {len(source_items)}")
display(source_items)


## 7. Inventário recursivo e controlado da fonte

O inventário identifica arquivos e diretórios disponíveis sem carregar datasets completos. A profundidade e a quantidade máxima são limitadas para evitar uma varredura excessiva.

O resultado será gravado apenas nos logs da Fase 3.


In [0]:
%python
def inventariar_path(root_path: str, max_depth: int = 3, max_items: int = 5000):
    registros = []
    pilha = [(root_path, 0)]

    while pilha and len(registros) < max_items:
        current_path, depth = pilha.pop()
        for item in dbutils.fs.ls(current_path):
            registros.append({
                "path": str(item.path),
                "name": str(item.name),
                "is_dir": str(bool(item.isDir())).lower(),
                "size_bytes": int(item.size),
                "depth": int(depth),
            })
            if item.isDir() and depth < max_depth:
                pilha.append((item.path, depth + 1))
            if len(registros) >= max_items:
                break

    return registros


inventory_records = inventariar_path(SOURCE_GOLD_PATH)

inventory_schema = StructType([
    StructField("path", StringType(), False),
    StructField("name", StringType(), False),
    StructField("is_dir", StringType(), False),
    StructField("size_bytes", LongType(), False),
    StructField("depth", LongType(), False),
])

df_gold_inventory = spark.createDataFrame(inventory_records, schema=inventory_schema)
display(df_gold_inventory.orderBy("depth", "path"))


## 8. Resumo dos formatos disponíveis

Este resumo ajuda a decidir como os próximos notebooks lerão cada dataset. Formatos Delta são identificados pela presença de `_delta_log`; arquivos Parquet, CSV e JSON são contabilizados por extensão.


In [0]:
%python
from pyspark.sql import functions as F

df_format_summary = (
    df_gold_inventory
    .filter(F.col("is_dir") == "false")
    .withColumn(
        "formato",
        F.when(F.lower("name").endswith(".parquet"), F.lit("parquet"))
         .when(F.lower("name").endswith(".csv"), F.lit("csv"))
         .when(F.lower("name").endswith(".json"), F.lit("json"))
         .when(F.col("path").contains("/_delta_log/"), F.lit("delta_log"))
         .otherwise(F.lit("outro"))
    )
    .groupBy("formato")
    .agg(F.count("*").alias("arquivos"), F.sum("size_bytes").alias("bytes"))
    .orderBy("formato")
)

display(df_format_summary)


## 9. Manifesto lógico dos datasets Gold

O manifesto usa os diretórios encontrados em tempo de execução, sem fixar nomes que podem ter evoluído na Fase 2. Ele registra onde cada ativo está e qual papel terá na Fase 3.


In [0]:
%python
gold_datasets = [
    {
        "dataset": item.name.rstrip("/"),
        "source_path": item.path.rstrip("/"),
        "access_mode": "READ_ONLY",
        "phase3_role": "candidate_ml_source",
    }
    for item in source_items
    if item.isDir()
]

if not gold_datasets:
    gold_datasets = [{
        "dataset": "gold_root",
        "source_path": SOURCE_GOLD_PATH,
        "access_mode": "READ_ONLY",
        "phase3_role": "candidate_ml_source",
    }]

display(spark.createDataFrame(gold_datasets).orderBy("dataset"))


## 10. Configuração central da Fase 3

O `config.json` centraliza caminhos, regras de separação e parâmetros iniciais. Os valores de modelagem são apenas defaults reproduzíveis e podem ser refinados depois da análise exploratória.

Para evitar data leakage, os conjuntos de treino, validação e teste terão destinos separados. O pipeline de pré-processamento deverá ser ajustado somente com dados de treino.


In [0]:
%python
config = {
    "project": {
        "name": PROJECT_NAME,
        "version": PROJECT_VERSION,
        "environment": ENVIRONMENT,
        "execution_ts_utc": EXECUTION_TS_UTC,
        "objective": "Prever se um aluno será alfabetizado ou não alfabetizado",
    },
    "environment": {
        "cloud": "AWS",
        "platform": "Databricks",
        "unity_catalog": True,
        "serverless_compatible": True,
        "external_volume": EXTERNAL_VOLUME,
        "volume_root": VOLUME_ROOT,
    },
    "source": {
        "phase": "tech_challenge_fase2",
        "layer": "gold",
        "access_mode": "READ_ONLY",
        "volume_path": SOURCE_GOLD_PATH,
        "s3_uri": SOURCE_GOLD_S3,
        "datasets": gold_datasets,
    },
    "destination": {
        "phase": PROJECT_VERSION,
        "base_path": BASE_PATH,
        "s3_uri": PROJECT_S3,
    },
    "paths": {
        "data": DATA_PATH,
        "input_gold_reference": INPUT_GOLD_PATH,
        "external": EXTERNAL_DATA_PATH,
        "interim": INTERIM_DATA_PATH,
        "processed": PROCESSED_DATA_PATH,
        "train": TRAIN_DATA_PATH,
        "validation": VALIDATION_DATA_PATH,
        "test": TEST_DATA_PATH,
        "notebooks": NOTEBOOKS_PATH,
        "src": SRC_PATH,
        "models": MODELS_PATH,
        "artifacts": ARTIFACTS_PATH,
        "reports": REPORTS_PATH,
        "images": IMAGES_PATH,
        "logs": LOGS_PATH,
        "config": CONFIG_PATH,
    },
    "repository": {
        "required_files": ["README.md", "requirements.txt", ".gitignore"],
        "required_src_modules": ["preprocessing", "modeling", "evaluation", "visualization"],
    },
    "machine_learning": {
        "problem_type": "binary_classification",
        "random_seed": 42,
        "split_strategy": "to_be_defined_after_eda",
        "default_split_ratio": {"train": 0.70, "validation": 0.15, "test": 0.15},
        "leakage_policy": "fit_preprocessing_only_on_train",
        "recommended_metrics": ["roc_auc", "precision", "recall", "f1", "confusion_matrix"],
        "interpretability": ["feature_importance", "shap"],
    },
}

config_file_path = f"{CONFIG_PATH}/config.json"
dbutils.fs.put(
    config_file_path,
    json.dumps(config, indent=2, ensure_ascii=False),
    overwrite=True,
)

print(f"Configuração salva em: {config_file_path}")


## 11. Validação do arquivo de configuração

A releitura confirma que os próximos notebooks poderão consumir o mesmo método de caminhos e segurança.


In [0]:
%python
config_loaded = json.loads(dbutils.fs.head(config_file_path))

assert config_loaded["source"]["access_mode"] == "READ_ONLY"
assert config_loaded["source"]["volume_path"] == SOURCE_GOLD_PATH
assert config_loaded["destination"]["base_path"] == BASE_PATH

display(config_loaded)


## 12. Persistência do inventário na Fase 3

O inventário cria uma evidência auditável dos ativos Gold disponíveis no momento do setup. A escrita ocorre exclusivamente na área de logs da Fase 3.


In [0]:
%python
inventory_output_path = (
    f"{LOGS_PATH}/setup/gold_inventory/execution_date={EXECUTION_DATE}"
)

(
    df_gold_inventory
    .write
    .mode("overwrite")
    .format("parquet")
    .option("compression", "snappy")
    .save(inventory_output_path)
)

print(f"Inventário salvo em: {inventory_output_path}")


## 13. Validação final da estrutura

A tabela final apresenta os caminhos críticos e seu estado. A validação da fonte é feita por leitura; a dos destinos confirma que os diretórios foram criados.


In [0]:
%python
validation_targets = [
    ("source_gold_fase2", SOURCE_GOLD_PATH, "READ_ONLY_SOURCE"),
    ("project_fase3", BASE_PATH, "READ_WRITE_DESTINATION"),
    ("data", DATA_PATH, "READ_WRITE_DESTINATION"),
    ("processed_train", TRAIN_DATA_PATH, "READ_WRITE_DESTINATION"),
    ("processed_validation", VALIDATION_DATA_PATH, "READ_WRITE_DESTINATION"),
    ("processed_test", TEST_DATA_PATH, "READ_WRITE_DESTINATION"),
    ("src", SRC_PATH, "READ_WRITE_DESTINATION"),
    ("models", MODELS_PATH, "READ_WRITE_DESTINATION"),
    ("artifacts", ARTIFACTS_PATH, "READ_WRITE_DESTINATION"),
    ("reports", REPORTS_PATH, "READ_WRITE_DESTINATION"),
    ("images", IMAGES_PATH, "READ_WRITE_DESTINATION"),
    ("config", CONFIG_PATH, "READ_WRITE_DESTINATION"),
]

validation_records = []
for name, path, role in validation_targets:
    try:
        dbutils.fs.ls(path)
        status, detail = "OK", ""
    except Exception as exc:
        status, detail = "ERROR", str(exc)[:500]
    validation_records.append({
        "name": name,
        "path": path,
        "role": role,
        "status": status,
        "detail": detail,
    })

df_setup_validation = spark.createDataFrame(validation_records)
display(df_setup_validation.orderBy("role", "name"))

errors = df_setup_validation.filter(F.col("status") != "OK").count()
if errors:
    raise RuntimeError(f"Setup finalizado com {errors} caminho(s) inválido(s).")

print("Setup da Fase 3 concluído com sucesso.")


## Resultado esperado

Ao final deste notebook:

- o External Volume existente foi validado;
- a Gold da Fase 2 foi conectada como fonte somente leitura;
- nenhum dado da Fase 2 foi alterado ou duplicado automaticamente;
- o prefixo `tech_challenge_fase3` foi criado de forma idempotente;
- os diretórios mínimos do desafio foram preparados;
- treino, validação e teste possuem áreas separadas;
- o inventário da Gold e o `config.json` foram gravados na Fase 3;
- a base está pronta para o notebook de inventário analítico e definição da variável-alvo.

### Próximo notebook sugerido

```text
01_gold_target_definition
```

Ele deverá inspecionar schemas e granularidade dos datasets Gold, validar a presença da variável-alvo de alfabetização, identificar chaves de integração e definir quais atributos podem entrar no modelo sem vazamento de dados.
